In [1]:
# Text splitter functionality is provided by LangChain framework
from langchain_text_splitters import HTMLHeaderTextSplitter, RecursiveCharacterTextSplitter

# Make use of BS for hadling the web content
import requests
from bs4 import BeautifulSoup

import lancedb

# Sentence transformers to use the embedding models locally
from sentence_transformers import SentenceTransformer, util
import pandas as pd

import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)


#### Utilities

> Library functions

**Custom Meta Data**  
From the parsed and split content, this function helps to create meta data in a custom way, that can be used while creating Knowledge DB

In [2]:
def meta_data_from_headings (heading: dict, n: int = 1, from_end: bool = True, sep: str = " : ") -> str:
    """
    Concatenates n values from a heading dictionary, either from the start or from the end.

    Param:
        heading (dict): Input dictionary for headings.
        n (int): Number of elements to take.
        from_end (bool): If True, take from the end; else from the start.
        sep (str): Optional separator to use between concatenated strings.

    Returns: Meta data as concatenation of headings.
    """
    values = list(heading.values())

    if n <= 0:
        n = 1
    if n > len(values):
        n = len(values)

    # Select n items from start or end
    selected = values[-n:] if from_end else values[:n]

    # Always concatenate in forward direction
    return sep.join(str(v) for v in selected)

**Get Main Content**

In [3]:
def get_main_content (url, type):

    html = requests.get(url).text
    soup = BeautifulSoup(html, "html.parser")

    # Remove layout elements
    for tag in soup(["nav", "header", "footer", "aside", "script", "style"]):
        tag.decompose()

    # Check and get main section of the pages
    main = soup.find("main")

    if not main:
        
        # fallback method, if no 'main' section in html page
        candidates = soup.find_all("div", recursive=True)
        main = max(candidates, key=lambda c: len(c.get_text(strip=True)), default=soup.body)

    # Get cleaned HTML content. Tags retained
    main_html = str(main)

    # If HTML content is required, provide with the tags
    if type == 'html':
        return (main_html)

    # If text is requirred, provide only the text content
    elif type == 'text':

        text_soup = BeautifulSoup (main_html, "html.parser")
        main_text = text_soup.get_text(separator="\n", strip=True)
        return main_text

**Multi-Pass Chunking**
> Often the scenario could be to incorporate multiple ways of chunking to have better granularity and meaning in the chunks  
> The Sentence and content aware chunkers are used in Combination to retain the context and be granular as well  
> The context is captured by the Meta data

In [4]:
# Define what are the splitters to be considered. There is default in library itself
seperators = [".", "?", "!"]

# Splitter function based on seperator and the length criteria
text_splitter = RecursiveCharacterTextSplitter (chunk_size=300, chunk_overlap=0,
                                                length_function=len, is_separator_regex=False,
                                                keep_separator=False,
                                                separators=seperators,
                                                )

# levels of header tags in html to split on
header_levels = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3"),
    ("h4", "Header 4"),
]

# Define a Splitter object for HTML content from the lib
# This library also gives splitter for Markdown, JSON etc
html_splitter = HTMLHeaderTextSplitter(header_levels)

**Combine 2 methods**  
Get the content and split it based on document structure first  
Some of the chunks can be big, because of the way the text is present  
Pass those blocks for one more level of splitting by sentences  
Capture the meta data from the headings and use it along with the text

In [5]:
def Build_Chunks (url, source, chunk_size_limit):

    # Get the main content
    HTML_Content = get_main_content (url, "html")

    # Chunk based on document structure
    docs = html_splitter.split_text (HTML_Content)

    # Start with empty list
    Chunks = []

    with open ('chunks.txt', mode='w') as f:

        for doc in docs :

            try :

                meta_data = meta_data_from_headings (doc.metadata)

                if not meta_data:
                    meta_data = 'Generic'

                # If the chunk is too long,
                if (len(doc.page_content) > chunk_size_limit):

                    # Split by sentece(s) by shorter lenth
                    splits = text_splitter.split_text(doc.page_content)

                    # Make them individual chunk with same meta data
                    for split in splits:

                        # Capture if the meta data and text are not the same
                        if (meta_data != split):

                            Chunk = {'source': source,'topic' : meta_data, 'text' : split}
                            print (Chunk, "\n----",file=f)

                            Chunks = Chunks + [Chunk]
                        
                else :
                    
                    if (meta_data != doc.page_content):
                        
                        Chunk = {'source': source, 'topic' : meta_data, 'text' : doc.page_content}
                        print (Chunk, "\n----",file=f)
                        Chunks = Chunks + [Chunk]
                
                # print (doc.metadata)
                # print ("Content : ", doc.page_content,"\n---")
                
            except Exception :
                pass

    print (len(Chunks))

    return Chunks

In [7]:
url = "https://www.ibm.com/think/topics/cloud-computing"

Chunks = Build_Chunks (url, "IBM", 500)

138


**Vectorise the data**  
> Once the chunks are creared along with the supporging data, use embedding model and craete vectors  
> Use a HF model which is suitable for general purpose 

In [8]:
Embedder = SentenceTransformer ("sentence-transformers/all-MiniLM-L6-v2")

The following layers were not sharded: encoder.layer.*.attention.self.key.weight, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.self.query.bias, encoder.layer.*.attention.output.LayerNorm.bias, embeddings.LayerNorm.weight, encoder.layer.*.attention.self.query.weight, encoder.layer.*.attention.self.value.weight, embeddings.word_embeddings.weight, encoder.layer.*.attention.self.key.bias, embeddings.LayerNorm.bias, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.output.dense.bias, pooler.dense.weight, encoder.layer.*.attention.self.value.bias, pooler.dense.bias, embeddings.token_type_embeddings.weight, encoder.layer.*.output.LayerNorm.weight, embeddings.position_embeddings.weight, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.attention.output.LayerNorm.weight


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
# Create vectors and store in the Chunks 
for idx, Chunk in enumerate (Chunks):

    vector = Embedder.encode (Chunk['text'])
    # print (type(vector))
    # print (vector)

    Chunks[idx]['vector'] = vector.tolist ()
  

In [10]:
Chunks

[{'source': 'IBM',
  'topic': 'Generic',
  'text': 'By ,  \nStephanie  Susnjara  \nIan Smalley  \nThis article was featured in the Think newsletter. .  \nGet it in your inbox',
  'vector': [0.04613076522946358,
   -0.0031102008651942015,
   0.013916854746639729,
   0.016215844079852104,
   0.08630746603012085,
   0.031712036579847336,
   -0.0480695478618145,
   0.05976236239075661,
   0.011187851428985596,
   0.06985259056091309,
   0.034592874348163605,
   0.03879470005631447,
   -0.07091591507196426,
   -0.05017615482211113,
   -0.005409654695540667,
   0.03182779625058174,
   0.02112996205687523,
   -0.01403551734983921,
   -0.08145879954099655,
   -0.006430749781429768,
   -0.08982961624860764,
   0.03827725350856781,
   -0.03279045596718788,
   0.029432624578475952,
   -0.0017526523442938924,
   -0.04319896176457405,
   0.0046217357739806175,
   0.0008705945801921189,
   -0.08916808664798737,
   0.010784988291561604,
   -0.0013256367528811097,
   0.01822863146662712,
   0.00144075

In [17]:
# Check various topics existing in all Chunks
Topics = list({c["topic"] for c in Chunks})
Topics

['Smart buildings',
 'Scale differentiated solutions',
 'Evolution of edge computing',
 'Authors',
 'What Are The Disadvantages Of Edge Computing?',
 'Cloud computing services',
 'Data processing',
 'What is quantum mechanics?',
 'Agentic AI versus generative AI',
 'Benefits of edge AI',
 'Smart devices',
 'Edge computing use cases',
 'What is Quantum computing?',
 'Decoherence',
 'Horizontal multi-agent',
 'Increase security',
 'Build with AWS IoT',
 'Understanding different computing types',
 'Manufacturing',
 'Edge computing for complex events',
 'Financial services',
 'Edge Computing vs Cloud Computing vs Fog Computing',
 'Unsupervised machine learning',
 'Data science versus business intelligence',
 'AWS Quantum Computing Next Steps',
 'The modern hybrid multicloud',
 'What are IoT technologies?',
 'Telecommunications',
 'Did you find what you were looking for today?',
 'Overfitting and underfitting',
 'Customer service automation',
 'Edge Computing: Redefining Digital Infrastruct

In [19]:
# Create a Lance DB Vector Base
DB = lancedb.connect ('Vector_DB')

# Create a Table and add the Chunks data
table = DB.create_table("article", data=Chunks, mode="overwrite") 
print (table.schema)

source: string
topic: string
text: string
vector: fixed_size_list<item: float>[768]
  child 0, item: float


In [20]:
# Query a vector
Query = "Platforms used as business in today's world"

Query_Vector = Embedder.encode (Query).tolist ()

Results = table.search(Query_Vector).limit(5).to_list ()

for Rs in Results :

    print (Rs['_distance']," ## ",Rs ['text'])

RuntimeError: lance error: Invalid user input: query dim(384) doesn't match the column vector vector dim(768), C:\Users\runneradmin\.cargo\registry\src\index.crates.io-1949cf8c6b5b557f\lance-4.0.0\src\dataset\scanner.rs:1421:32

**Create a Tech Repo**  
> From various sources in internet, create a knowledge repo with all information chunked and vectorised

In [21]:
# Gather multiple reference material for Technology information
References = [{'Source' : 'IBM', 'url' : "https://www.ibm.com/think/topics/cloud-computing"},
              {'Source' : 'Oracle', 'url' : "https://www.oracle.com/in/cloud/what-is-cloud-computing/"},
              {'Source' : 'AWS', 'url' : "https://aws.amazon.com/what-is/iot/"},
              {'Source' : 'IBM', 'url' : "https://www.ibm.com/think/topics/edge-ai"},              
              {'Source' : 'Microsoft', 'url' : "https://azure.microsoft.com/en-us/resources/cloud-computing-dictionary/what-is-edge-computing"},
              {'Source' : 'IBM', 'url' : "https://www.ibm.com/think/topics/edge-computing"},              
              {'Source' : 'Fortinet', 'url' : "https://www.fortinet.com/resources/cyberglossary/edge-computing"},
              {'Source' : 'NVIDIA', 'url' : "https://blogs.nvidia.com/blog/what-is-edge-ai/"},
              {'Source' : 'MIT', 'url' : "https://mitsloan.mit.edu/ideas-made-to-matter/machine-learning-explained"},
              {'Source' : 'AWS', 'url' : "https://aws.amazon.com/what-is/machine-learning/"},
              {'Source' : 'AWS', 'url' : "https://aws.amazon.com/what-is/quantum-computing/"},
              {'Source' : 'Caltech', 'url' : "https://scienceexchange.caltech.edu/topics/quantum-science-explained/quantum-computing-computers"},
              {'Source' : 'MIT', 'url' : "https://mitsloan.mit.edu/ideas-made-to-matter/quantum-computing-what-leaders-need-to-know-now"},
              {'Source' : 'IBM', 'url' : "https://www.ibm.com/think/topics/data-science"},
              {'Source' : 'Berkley', 'url' : "https://ischoolonline.berkeley.edu/data-science/what-is-data-science/"},
              {'Source' : 'MIT', 'url' : "https://mitsloan.mit.edu/ideas-made-to-matter/agentic-ai-explained"},
              {'Source' : 'Google', 'url' : "https://cloud.google.com/discover/what-is-agentic-ai"},
              {'Source' : 'AWS', 'url' : "https://aws.amazon.com/what-is/agentic-ai/"},
              {'Source' : 'Accenture', 'url' : "https://talentsprint.com/blog/what-is-agentic-ai-guide"}
              
            ]

Chunks = []
for Ref in References:
    
    Parts = Build_Chunks (Ref['url'], Ref ['Source'], 500)
    Chunks = Chunks + Parts

print (len(Chunks))

138
133
47
63
76
62
83
47
90
94
55
34
41
74
49
69
37
81
57
1330


In [13]:
Embedder_1 = SentenceTransformer ("sentence-transformers/all-mpnet-base-v2")

The following layers were not sharded: encoder.layer.*.attention.attn.o.bias, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.attn.k.bias, encoder.layer.*.attention.attn.v.bias, embeddings.LayerNorm.weight, encoder.relative_attention_bias.weight, encoder.layer.*.attention.LayerNorm.weight, embeddings.word_embeddings.weight, encoder.layer.*.attention.attn.q.bias, encoder.layer.*.attention.attn.v.weight, embeddings.LayerNorm.bias, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.attention.attn.o.weight, encoder.layer.*.output.dense.bias, pooler.dense.weight, pooler.dense.bias, encoder.layer.*.attention.LayerNorm.bias, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.attention.attn.k.weight, embeddings.position_embeddings.weight, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.attention.attn.q.weight


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [14]:
# Create vectors and store in the Chunks 
for idx, Chunk in enumerate (Chunks):

    vector = Embedder_1.encode (Chunk['text'])
    Chunks[idx]['vector'] = vector.tolist ()

In [15]:
# Create a Table and add the Chunks data
table = DB.create_table("tech_ref", data=Chunks, mode="overwrite") 
print (table.schema)

source: string
topic: string
text: string
vector: fixed_size_list<item: float>[768]
  child 0, item: float


In [16]:
# Query a vector
# Query = "Classic computing changes its shape in different forms"
Query = "Edge computing does not live on the edge"

Query_Vector = Embedder_1.encode (Query).tolist ()

Results = table.search(Query_Vector).distance_type("cosine").limit(5).to_list ()
# Results = table.search().where("topic IN ('Multicloud')").to_list ()

for Rs in Results :

    # print (Rs['_distance'],Rs['source']," ## ",Rs ['text'])
    print (Rs['source']," ## ",Rs ['text'])

Microsoft  ##  Edge computing processes data where it's created—at the “edge” of the network—rather than sending all the unstructured information to distant datacenters
IBM  ##  In contrast to , which relies on remote access to computing resources like compute, storage and over the internet, edge computing processes data locally where devices gather it. While distinctly different, edge computing extends the functions of the cloud model to edge locations
Fortinet  ##  Because telecommunications organizations help companies set up networks, they rely on edge computing topology to enable a wide range of devices to connect to the organization’s network and function near its edge
Microsoft  ##  Edge computing extends beyond traditional IT infrastructure and helps reshape how organizations capture value from distributed data. By processing information at the remote borders of the network—rather than distant datacenters—this technology enables millisecond responses, reduces costs, and unlocks